# E25 — o peso de cada explicação

O capítulo 9 responde **quantas** peças da família reproduzem as quatro estatísticas do dado: duas.
Ele não responde **com que folga** cada uma cabe, e é isso que separa um empate de uma ordem com
folga pequena.

A folga de uma peça é a distância à fronteira da tolerância, medida na unidade de cada estatística:
1 é estar na borda, 0,5 é ter metade do que se aceita.

In [1]:
# <- brinque com: SERIE, SEMENTE, GRADE_P, GRADE_RAZAO, GRADE_PERMANENCIA
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import frevolab
from frevolab import evidencia, graficos

RAIZ = Path.cwd()
SERIE, SEMENTE = evidencia.SERIE_PADRAO, evidencia.SEMENTE_PADRAO

pecas, real = evidencia.familia(SERIE, SEMENTE)
cabem = evidencia.as_que_cabem(pecas)
ranking = evidencia.margens(pecas, 8)

print("frevolab %s | %s | %d pecas na familia | cabem na tolerancia: %d"
      % (frevolab.VERSAO, SERIE, len(pecas), len(cabem)))
print("as quatro estatisticas do dado: %s" % {k: round(v, 4) for k, v in real.items() if k in evidencia.CHAVES_PADRAO})
print()
tabela = pd.DataFrame(ranking).set_index("folga")
print(tabela[["p", "razao", "permanencia", "quem_manda"]].round(4).to_string())

frevolab 0.1.0 | sp500.csv | 180 pecas na familia | cabem na tolerancia: 2
as quatro estatisticas do dado: {'taxa': 0.0513, 'pior': 20, 'mediana': 2.0, 'acima_do_dobro': 0.1305}

             p  razao  permanencia      quem_manda
folga                                             
0.967692  0.05    4.0         30.0  acima_do_dobro
1.000000  0.25    3.0         60.0            pior
1.092555  0.08    2.5         60.0  acima_do_dobro
1.222621  0.12    3.0        120.0  acima_do_dobro
1.378700  0.08    3.5         30.0  acima_do_dobro
1.469224  0.12    4.0         60.0            taxa
1.500000  0.08    4.0         30.0            pior
1.500000  0.12    2.5         30.0            pior


In [2]:
# Figura 1: a folga de cada peça, da que mais tem a favor para a que menos tem.
folgas = np.sort([c["folga"] for c in pecas])
fig, eixo = plt.subplots(figsize=(8.6, 4.2))
eixo.plot(range(1, len(folgas) + 1), folgas, lw=1.8, color="#1f4e79")
eixo.axhline(1.0, color="#b03a2e", ls="--", lw=1.4, label="a fronteira da tolerância")
eixo.scatter([1, 2], folgas[:2], s=70, color="#2e7d32", zorder=5,
             label="as %d que cabem" % len(cabem))
eixo.axvspan(0.5, len(cabem) + 0.5, color="#2e7d32", alpha=0.12)
eixo.set_xlabel("peças da família, da que mais tem a favor para a que menos tem")
eixo.set_ylabel("folga: distância à fronteira, na unidade da tolerância")
eixo.set_ylim(0.9, min(2.2, folgas.max()))
eixo.legend(frameon=False, fontsize=9)
eixo.grid(alpha=0.25)
fig.tight_layout()
graficos.salvar(fig, "E25_evidencia", 1)
plt.close(fig)
print("a peca mais distante da familia esta a %.2f tolerancias" % folgas.max())

a peca mais distante da familia esta a 6.50 tolerancias


## Leitura visual das figuras

Feita nesta sessão abrindo o .png com a ponte de visão (AGENTS.md §9), depois de o caderno rodar.

O que o desenho mostra: a curva das folgas sobe depressa nos primeiros pontos e depois se achata,
mas **sai do quadro** por volta da peça de número quarenta --- o eixo vertical foi cortado em 2,2
tolerâncias para que a região que decide apareça, e a peça mais distante da família está a 6,5
tolerâncias. A faixa verde marca as duas primeiras, que são as que o capítulo conta, e a linha
tracejada é a fronteira: as duas estão encostadas nela, e a terceira fica visivelmente acima.

In [3]:
# O resultado: um objeto por grandeza, para o livro citar por comando.
resultado = {
    "evidencia_pecas": int(len(pecas)),
    "evidencia_cabem": int(len(cabem)),
    "evidencia_folga_primeira": float(ranking[0]["folga"]),
    "evidencia_folga_segunda": float(ranking[1]["folga"]),
    "evidencia_folga_terceira": float(ranking[2]["folga"]),
    "evidencia_folga_distante": float(max(c["folga"] for c in pecas)),
    "evidencia_razao_duas": float(ranking[1]["folga"] / ranking[0]["folga"]),
    "evidencia_razao_fronteira": float(ranking[2]["folga"] / ranking[1]["folga"]),
    "evidencia_semente": int(SEMENTE),
    "evidencia_tolerancias": int(len(evidencia.CHAVES_PADRAO)),
}
caminho = Path("lab/resultados/E25_evidencia.json")
caminho.write_text(json.dumps(resultado, indent=1, ensure_ascii=False, sort_keys=True), encoding="utf-8")
print("%s gravado | %d grandezas" % (caminho, len(resultado)))

lab/resultados/E25_evidencia.json gravado | 10 grandezas
